In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. import numpy, pandas, matplotlib, seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 2. matplotlib inline 설정
%matplotlib inline

In [3]:
data_processed_parquet_path = "drive/MyDrive/Semiconductor-AnomalyDetection/data/processed/master_df.parquet"
master_df = pd.read_parquet(data_processed_parquet_path)

In [6]:
#Baseline goal
# 현재 데이터로 예측 가능한가?
# 가장 기본 모델로 어느 정도 성능이 나오나?
# 이후 RF, XGBoost 같은 더 복잡한 모델이 baseline보다 좋아지는가?

# master_df 불러오기
# → X(입력), y(정답) 나누기
# → train/test 나누기
# → train으로 전처리+학습
# → test로 예측
# → 성능 평가

In [ ]:
import os
import json # 파일 읽고 쓰는 library
import joblib # 모델 load/save library


from sklearn.model_selection import train_test_split #data split to train and test
from sklearn.pipeline import Pipeline #여러 단계 묶기
from sklearn.impute import SimpleImputer # 결측치 채우는 도구
from sklearn.preprocessing import StandardScaler #Feature Scaling mean:0, var:1 값 범위 맞추기
from sklearn.linear_model import LogisticRegression #baseline model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)


In [ ]:
#1. setting
DATA_PATH = "data/processed/master_df.parquet"
TARGET_COL = "label"   # 네 실제 타겟 컬럼명으로 바꿔
DROP_COLS = ["sample_id", "timestamp"]  # 없으면 지워도 됨
RANDOM_STATE = 42
TEST_SIZE = 0.2

RESULT_DIR = "results/baseline"
MODEL_DIR = "models"

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
#2. data loading
df = pd.read_parquet(DATA_PATH)
print(f"Loaded data shape: {df.shape}")

# 필요 없는 컬럼 제거
drop_cols_exist = [col for col in DROP_COLS if col in df.columns]
df = df.drop(columns=drop_cols_exist, errors="ignore")

# 타겟 존재 확인
if TARGET_COL not in df.columns:
    raise ValueError(f"'{TARGET_COL}' column not found in dataframe.")

# X, y 분리
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].copy()

print("Target distribution:")
print(y.value_counts(dropna=False))

In [ ]:



# =========================
# 3. 타겟 라벨 확인 / 변환
# =========================
# 만약 현재 라벨이 -1, 1 이라면 그대로 logistic regression 가능
# 다만 평가할 때 positive class를 1로 둘 거라면 아래처럼 정리해두면 좋음.
unique_labels = sorted(y.dropna().unique().tolist())
print("Unique labels:", unique_labels)

# 예: 혹시 1 / -1 구조면 그대로 사용
# 예: 혹시 문자열이면 숫자로 바꿔야 함
# 필요시 아래처럼 사용
# y = y.map({"normal": 0, "fault": 1})

if y.isna().sum() > 0:
    raise ValueError("Target column contains missing values. Handle target NaNs first.")


# =========================
# 4. 숫자형 feature만 사용
# =========================
# baseline에서는 숫자형 컬럼만 먼저 사용하자
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X = X[numeric_cols]

print(f"Number of numeric features: {len(numeric_cols)}")

if len(numeric_cols) == 0:
    raise ValueError("No numeric columns found for baseline model.")


# =========================
# 5. Train / Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


# =========================
# 6. Pipeline 정의
# =========================
# baseline에서는:
# 결측치 대체 -> scaling -> logistic regression
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight="balanced"   # 불균형 데이터면 baseline에서 유용
    ))
])


# =========================
# 7. 학습
# =========================
pipeline.fit(X_train, y_train)


# =========================
# 8. 예측
# =========================
y_pred = pipeline.predict(X_test)

# roc_auc용 probability
# logistic regression은 predict_proba 사용 가능
y_proba = pipeline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None


# =========================
# 9. 평가
# =========================
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    "recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "f1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
}

if y_proba is not None:
    metrics["roc_auc"] = roc_auc_score(y_test, y_proba)

cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)

print("\n=== Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

print("\n=== Confusion Matrix ===")
print(cm)

print("\n=== Classification Report ===")
print(report)


# =========================
# 10. 결과 저장
# =========================
with open(os.path.join(RESULT_DIR, "logistic_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

with open(os.path.join(RESULT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

np.savetxt(
    os.path.join(RESULT_DIR, "confusion_matrix.txt"),
    cm,
    fmt="%d"
)

joblib.dump(pipeline, os.path.join(MODEL_DIR, "logistic_baseline.pkl"))

print("\nSaved metrics, report, confusion matrix, and model.")